# Bằng chứng định tính — chạy bằng tài khoản Google khác
Chọn T4 GPU rồi Run All. Notebook tự clone repo và tải ZIP công khai từ thư mục Drive đã chia sẻ; không cần mount Drive/upload ZIP và không train lại. Tải ZIP bằng chứng ở ô cuối, giải nén vào repo local. Nếu Drive báo quota, chờ hoặc tải ZIP từ link Drive rồi đặt /content/lab21_shared_backup.zip.


In [ ]:
# Setup Colab web — repo của học viên
from pathlib import Path
import os,sys,subprocess
ROOT=Path("/content/Day21-Track3-Finetuning-Lab")
REPO="https://github.com/HuyHaiThanh/K4-Track3A-DAY21-NguyenVanHuy-2A202602428-Finetuning-Lab.git"
if not ROOT.exists():
    subprocess.run(["git","clone",REPO,str(ROOT)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(ROOT / "requirements.txt")],check=True)
os.chdir(ROOT)
import torch
assert torch.cuda.is_available(), "Chọn T4 GPU trước khi Run All."


In [ ]:
# Tự khôi phục backup công khai — tài khoản khác không cần mount Drive.
from pathlib import PurePosixPath
from zipfile import ZipFile,is_zipfile
import shutil
BACKUP_FILE_ID = "1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p"
BACKUP_FOLDER_URL = "https://drive.google.com/drive/folders/18wN-UeRbCC6YQk1lYVV7v0HTdt8wrSOk"
BACKUP_SIZE = 481443440
required = ["results/verdict.json","results/baselines_frozen.json","data/split/train.jsonl",
            "adapters/correct/adapter_model.safetensors","adapters/correct/adapter_config.json"]
if not all((ROOT / name).is_file() for name in required):
    subprocess.run([sys.executable,"-m","pip","install","-q","gdown>=5,<6"],check=True)
    import gdown
    archive_path=Path("/content/lab21_shared_backup.zip")
    if not archive_path.is_file() or archive_path.stat().st_size != BACKUP_SIZE:
        downloaded=gdown.download(id=BACKUP_FILE_ID,output=str(archive_path),quiet=False)
        assert downloaded, "Không tải được ZIP: kiểm tra quyền Anyone with the link hoặc giới hạn tải của Drive."
    assert archive_path.stat().st_size == BACKUP_SIZE, "ZIP chưa tải đủ hoặc file Drive đã thay đổi."
    assert is_zipfile(archive_path), "Nội dung tải về không phải ZIP."
    with ZipFile(archive_path) as archive:
        assert archive.testzip() is None, "ZIP bị lỗi CRC."
        for entry in archive.infolist():
            parts=PurePosixPath(entry.filename).parts
            assert ".." not in parts and not PurePosixPath(entry.filename).is_absolute(), "ZIP có đường dẫn không an toàn."
            positions=[i for i,x in enumerate(parts) if x in ("results","adapters","data")]
            if not positions: continue
            relative=PurePosixPath(*parts[positions[0]:])
            allowed=relative.parts[0] == "results" or relative.parts[:2] in (("data","split"),("adapters","correct"))
            if not allowed or entry.is_dir(): continue
            assert (entry.external_attr >> 16) & 0o170000 != 0o120000, "ZIP có symlink."
            target=(ROOT / str(relative)).resolve()
            assert ROOT.resolve() in target.parents
            target.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(entry) as source,target.open("wb") as dest:
                shutil.copyfileobj(source,dest)
assert all((ROOT / name).is_file() for name in required), "Backup thiếu artifact cần thiết."
print("Đã có baseline và adapter correct. Bước tiếp theo chỉ inference, không train lại.")


In [ ]:
from pathlib import Path
import os,sys,subprocess,base64
ROOT=Path("/content/Day21-Track3-Finetuning-Lab")
assert (ROOT / "results/verdict.json").is_file(), "Dùng runtime đã hoàn thành lab; nếu là máy mới, khôi phục ZIP vào ROOT trước."
assert (ROOT / "adapters/correct/adapter_model.safetensors").is_file()
os.chdir(ROOT)
(ROOT / "scripts/collect_qualitative.py").write_bytes(base64.b64decode("IiIiQ29sbGVjdCBmdWxsIHF1YWxpdGF0aXZlIGV2aWRlbmNlIHdpdGhvdXQgdHJhaW5pbmcgb3IgY2hhbmdpbmcgZnJvemVuIG1ldHJpY3MuIiIiCmltcG9ydCBoYXNobGliLCBqc29uLCBwYXRobGliLCBzeXMKUk9PVCA9IHBhdGhsaWIuUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0Kc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFJPT1QgLyAic3JjIikpCmZyb20gbGFia2l0IGltcG9ydCBldmFsdWF0ZSBhcyBldiwgZ2VuZXJhdGUsIHJlcG9ydApmcm9tIGxhYmtpdC5jb25maWcgaW1wb3J0IGdldF90aWVyCgpkZWYgbWFpbigpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHBlZnQgaW1wb3J0IFBlZnRNb2RlbAogICAgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCksICJDaOG6oXkgdHLDqm4ga2VybmVsIENvbGFiIEdQVS4iCiAgICBmcm96ZW5fcGF0aCA9IFJPT1QgLyAicmVzdWx0cy9iYXNlbGluZXNfZnJvemVuLmpzb24iCiAgICB2ZXJkaWN0X3BhdGggPSBST09UIC8gInJlc3VsdHMvdmVyZGljdC5qc29uIgogICAgYmVmb3JlID0ge3N0cihwKTogaGFzaGxpYi5zaGEyNTYocC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpIGZvciBwIGluIFtmcm96ZW5fcGF0aCx2ZXJkaWN0X3BhdGhdfQogICAgZnJvemVuID0ganNvbi5sb2Fkcyhmcm96ZW5fcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICB2ZXJkaWN0ID0ganNvbi5sb2Fkcyh2ZXJkaWN0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgdGllciA9IGdldF90aWVyKGZyb3plblsidGllciJdKQogICAgYXNzZXJ0IHRpZXIubW9kZWxfaWQgPT0gZnJvemVuWyJtb2RlbCJdCiAgICBhc3NlcnQgbm90IGZyb3plbi5nZXQoInNtb2tlX21vZGUiKQogICAgYXNzZXJ0IGhhc2hsaWIuc2hhMjU2KGdlbmVyYXRlLk9QVElNSVpFRF9QUk9NUFQuZW5jb2RlKCkpLmhleGRpZ2VzdCgpWzoxNl0gPT0gZnJvemVuWyJvcHRpbWl6ZWRfcHJvbXB0X3NoYSJdCiAgICB0YXJnZXQgPSBbanNvbi5sb2FkcyhsKSBmb3IgbCBpbiAoUk9PVCAvICJkYXRhL2V2YWxfdGFyZ2V0Lmpzb25sIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCldCiAgICBhc3NlcnQgbGVuKHRhcmdldCkgPT0gZnJvemVuWyJuX3RhcmdldCJdID09IDUwCiAgICByZWZlcmVuY2VzID0ganNvbi5sb2FkcygoUk9PVCAvICJkYXRhL2NoZWNrc3Vtcy5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgcmF3ID0gKFJPT1QgLyAiZGF0YS9ldmFsX3RhcmdldC5qc29ubCIpLnJlYWRfYnl0ZXMoKQogICAgYXNzZXJ0IGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KClbOjE2XSA9PSByZWZlcmVuY2VzWyJldmFsX3RhcmdldC5qc29ubCJdLCAiRXZhbCBjaGVja3N1bSBraMOhYy4iCiAgICBtb2RlbCx0b2sgPSBnZW5lcmF0ZS5sb2FkX2Jhc2UodGllcikKICAgIHByZWRzX2IsXyA9IGdlbmVyYXRlLmdlbmVyYXRlX2JhdGNoKG1vZGVsLHRvayxbclsiaW5wdXQiXSBmb3IgciBpbiB0YXJnZXRdLHN5c3RlbT1nZW5lcmF0ZS5PUFRJTUlaRURfUFJPTVBULGxhYmVsPSJxdWFsaXRhdGl2ZS9iYXNlLWIiKQogICAgZGVsIG1vZGVsCiAgICBnZW5lcmF0ZS5mcmVlX21lbW9yeSgpCiAgICBtb2RlbCx0b2sgPSBnZW5lcmF0ZS5sb2FkX2Jhc2UodGllcikKICAgIG1vZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCxzdHIoUk9PVCAvICJhZGFwdGVycy9jb3JyZWN0IikpCiAgICBtb2RlbC5ldmFsKCkKICAgIHByZWRzX2Z0LF8gPSBnZW5lcmF0ZS5nZW5lcmF0ZV9iYXRjaChtb2RlbCx0b2ssW3JbImlucHV0Il0gZm9yIHIgaW4gdGFyZ2V0XSxzeXN0ZW09Z2VuZXJhdGUuTkFJVkVfUFJPTVBULGxhYmVsPSJxdWFsaXRhdGl2ZS9mdCIpCiAgICBkZWwgbW9kZWwKICAgIGdlbmVyYXRlLmZyZWVfbWVtb3J5KCkKICAgIGV4YW1wbGVzPVtdCiAgICBmb3IgaSwocm93LGIsZnQpIGluIGVudW1lcmF0ZSh6aXAodGFyZ2V0LHByZWRzX2IscHJlZHNfZnQpKToKICAgICAgICBzYj1ldi50cmlhZ2VfZmllbGRfYWNjdXJhY3koYixyb3dbImxhYmVsIl0pCiAgICAgICAgc2Y9ZXYudHJpYWdlX2ZpZWxkX2FjY3VyYWN5KGZ0LHJvd1sibGFiZWwiXSkKICAgICAgICBleGFtcGxlcy5hcHBlbmQoeyJpIjppLCJ0aWNrZXQiOnJvd1siaW5wdXQiXSwibGFiZWwiOnJvd1sibGFiZWwiXSwicHJlZF9iIjpiLCJwcmVkX2Z0IjpmdCwic2NvcmVfYiI6c2IsInNjb3JlX2Z0IjpzZiwiZGVsdGEiOnNmLXNifSkKICAgIG9ic2VydmVkX2I9c3VtKHJbInNjb3JlX2IiXSBmb3IgciBpbiBleGFtcGxlcykvbGVuKGV4YW1wbGVzKQogICAgb2JzZXJ2ZWRfZnQ9c3VtKHJbInNjb3JlX2Z0Il0gZm9yIHIgaW4gZXhhbXBsZXMpL2xlbihleGFtcGxlcykKICAgIGV4cGVjdGVkX2Z0PW5leHQoclsidGFyZ2V0Il0gZm9yIHIgaW4gdmVyZGljdFsiY29tcGFyaXNvbiJdIGlmICIoYykiIGluIHJbInJ1biJdKQogICAgbWF0Y2hlcz1hYnMob2JzZXJ2ZWRfYi1mcm96ZW5bImJhc2VsaW5lX2IiXVsidGFyZ2V0Il0pPDFlLTYgYW5kIGFicyhvYnNlcnZlZF9mdC1leHBlY3RlZF9mdCk8MWUtNgogICAgcmVwb3J0LndyaXRlX2pzb24oeyJzdXBwbGVtZW50YXJ5X2luZmVyZW5jZSI6VHJ1ZSwiYmFzZWxpbmVfdGFyZ2V0IjpvYnNlcnZlZF9iLCJmdF90YXJnZXQiOm9ic2VydmVkX2Z0LCJtYXRjaGVzX2Zyb3plbl9zY29yZXMiOm1hdGNoZXN9LCJxdWFsaXRhdGl2ZV9yZXByb2R1Y3Rpb25fY2hlY2suanNvbiIscmVzdWx0c19kaXI9Uk9PVCAvICJyZXN1bHRzIikKICAgIHJlcG9ydC53cml0ZV9qc29uKGV4YW1wbGVzLCJxdWFsaXRhdGl2ZV9jb21wYXJpc29uLmpzb24iLHJlc3VsdHNfZGlyPVJPT1QgLyAicmVzdWx0cyIpCiAgICBsb3NzZXM9c29ydGVkKFtyIGZvciByIGluIGV4YW1wbGVzIGlmIHJbImRlbHRhIl08MF0sa2V5PWxhbWJkYSByOnJbImRlbHRhIl0pCiAgICB3aW5zPXNvcnRlZChbciBmb3IgciBpbiBleGFtcGxlcyBpZiByWyJkZWx0YSJdPjBdLGtleT1sYW1iZGEgcjotclsiZGVsdGEiXSkKICAgIHNlbGVjdGVkPWxvc3Nlc1s6Ml0rd2luc1s6Ml0KICAgIGZvciByIGluIHNvcnRlZChleGFtcGxlcyxrZXk9bGFtYmRhIHI6clsic2NvcmVfZnQiXSk6CiAgICAgICAgaWYgbGVuKHNlbGVjdGVkKT49NTogYnJlYWsKICAgICAgICBpZiByIG5vdCBpbiBzZWxlY3RlZDpzZWxlY3RlZC5hcHBlbmQocikKICAgIHJlcG9ydC53cml0ZV9qc29uKHNlbGVjdGVkLCJxdWFsaXRhdGl2ZV9zZWxlY3RlZC5qc29uIixyZXN1bHRzX2Rpcj1ST09UIC8gInJlc3VsdHMiKQogICAgbGluZXM9WyIjIELhurFuZyBjaOG7qW5nIMSR4buLbmggdMOtbmggYuG7lSBzdW5nIiwgIiIsIGYiRlQgdGjhuq9uZzoge2xlbih3aW5zKX07IHRodWE6IHtsZW4obG9zc2VzKX07IGjDsmE6IHtsZW4oZXhhbXBsZXMpLWxlbih3aW5zKS1sZW4obG9zc2VzKX0uIiwgZiLEkGnhu4NtIHNpbmggbOG6oWkga2jhu5twIGvhur90IHF14bqjIMSRw7NuZyBixINuZzoge21hdGNoZXN9LiIsICIiXQogICAgZm9yIHIgaW4gc2VsZWN0ZWQ6CiAgICAgICAgbGluZXMgKz0gW2YiIyMgVGlja2V0IHtyWydpJ119IixyWyd0aWNrZXQnXSwiIiwgIk5ow6NuOiAiK2pzb24uZHVtcHMoclsnbGFiZWwnXSxlbnN1cmVfYXNjaWk9RmFsc2UpLCIiLCAiQmFzZWxpbmUgKGIpOiIsICJ+fn5qc29uIixyWydwcmVkX2InXSwifn5+IiwgIkZpbmUtdHVuZToiLCJ+fn5qc29uIixyWydwcmVkX2Z0J10sIn5+fiIsZiLEkGnhu4NtIChiKT17clsnc2NvcmVfYiddfTsgRlQ9e3JbJ3Njb3JlX2Z0J119OyBkZWx0YT17clsnZGVsdGEnXX0uIiwgIiJdCiAgICBpZiBsZW4obG9zc2VzKTwyOmxpbmVzICs9IFsiS2jDtG5nIMSR4bunIGhhaSBjYSBGVCB0aHVhIHRyw6puIHThuq1wIGV2YWwgY+G7kSDEkeG7i25oLiBLaGFpIGLDoW8gaOG6oW4gY2jhur8gbsOgeTsga2jDtG5nIHThuqFvIGNhIHRodWEgZ2nhuqMuIl0KICAgIGlmIG5vdCBtYXRjaGVzOmxpbmVzICs9IFsixJBp4buDbSBzaW5oIGzhuqFpIGtow7RuZyBraOG7m3A6IMSRw6J5IGzDoCBr4bq/dCBxdeG6oyBi4buVIHN1bmcsIGNoxrBhIHRo4buDIGTDuW5nIHRoYXkgY2hvIMSR4bqndSByYSBs4bqnbiDEkW8gY2jDrW5oLiBLaeG7g20gdHJhIHBoacOqbiBi4bqjbi9nZW5lcmF0aW9uIHRyxrDhu5tjIGtoaSBr4bq/dCBsdeG6rW4uIl0KICAgIChST09UIC8gInN1Ym1pc3Npb24vUVVBTElUQVRJVkVfRVZJREVOQ0UubWQiKS53cml0ZV90ZXh0KCJcbiIuam9pbihsaW5lcyksZW5jb2Rpbmc9InV0Zi04IikKICAgIGZvciBwLGggaW4gYmVmb3JlLml0ZW1zKCk6YXNzZXJ0IGhhc2hsaWIuc2hhMjU2KHBhdGhsaWIuUGF0aChwKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpPT1oCiAgICBwcmludChmIsSQw6MgbMawdSDEkeG7pyDEkeG6p3UgcmEuIEJhc2VsaW5lPXtvYnNlcnZlZF9ifSwgRlQ9e29ic2VydmVkX2Z0fSwga2jhu5twPXttYXRjaGVzfTsgY2EgdGh1YT17bGVuKGxvc3Nlcyl9LiIpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOm1haW4oKQo="))
os.environ["COMPUTE_TIER"]="T4"
os.environ.pop("BASE_MODEL",None)
os.environ.pop("EVAL_LIMIT",None)
subprocess.run([sys.executable,"-u","scripts/collect_qualitative.py"],cwd=ROOT,check=True)


In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
archive=ROOT.parent / "lab21_qualitative_evidence.zip"
paths=["results/qualitative_comparison.json","results/qualitative_selected.json","results/qualitative_reproduction_check.json","submission/QUALITATIVE_EVIDENCE.md"]
with ZipFile(archive,"w",ZIP_DEFLATED) as z:
    for name in paths:z.write(ROOT / name,name)
print("ZIP:",archive)
# Colab web có thể tải tự động; VS Code giữ ZIP trên runtime.
if type(get_ipython()).__module__.startswith("google.colab"):
    from google.colab import files
    files.download(str(archive))
